# Music Annotation Project — Notebook 71R1
## Exact Notebook 70/Pilot 09 roster: fresh Notebook 09 vs fresh V8.8 whole-page scoring

**Runtime:** CPU. Run all cells in order.

This replacement fixes Notebook 71's cohort and evaluation defects. It uses the exact Notebook 70 page roster, the exact Pilot 09 observed pages and reviewed masks, and the exact Notebook 53 registered-replay V8.8 lineage. Both engines are rebuilt before clean truth is loaded. Scoring is performed once per page over the complete page and inside the shared authoritative M3 boundary.

### Fixed comparison

- **Notebook 09:** fresh Telea on M3, then fresh canonical `geometry_t80`.
- **V8.8:** fresh registered-replay R2 Telea on `M5_REMOVE = M3 & ~SAFE_REMOVE`; SAFE_REMOVE retains exact observed bytes.
- **Shared authority:** M3 is identical for both paths and V8.8 can only subtract from it.
- **Primary question:** does V8.8 reduce missing genuine ink without increasing false inserted ink/residual annotation?
- **Truth firewall:** clean pages are loaded only after both candidate pages are written and hashed.
- **No production authority.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-image matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, platform, shutil, sys, zipfile
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from skimage.metrics import structural_similarity

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NOTEBOOK70_RUN_OVERRIDE=None
NOTEBOOK09_RUN_OVERRIDE=None
NOTEBOOK53_RUN_OVERRIDE=None
N70_ROOTS=[ROOT/'notebook70_whole_page_crossover',ROOT/'notebook70',ROOT/'whole_page_crossover_notebook70']
N09_ROOT=ROOT/'production_pilot'/'runs'
N53_ROOTS=[ROOT/'observed_print_classifier_v8_8b',ROOT/'v8_8b_operational_builder',ROOT/'notebook53_v8_8b']
CLEAN_DIR=ROOT/'source'/'clean'
OUTPUT_ROOT=ROOT/'notebook71R1_whole_page_engine_comparison'
MASK_DILATION=2;CROP_MARGIN=48;GEOMETRY_THRESHOLD=80;BAND_WIDTH=4
BOOTSTRAP_RESAMPLES=10000;SEED=71001;MAX_VISUAL_PAGES=12
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook71R1'
OUT=OUTPUT_ROOT/RUN_ID;INPUTS=OUT/'inputs';PAGES_OUT=OUT/'pages';TABLES=OUT/'tables';AUDIT=OUT/'audit';VISUAL=OUT/'visual_audit';UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,PAGES_OUT,TABLES,AUDIT,VISUAL,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)

## 2. Helpers

In [ ]:
def require(c,m):
    if not c: raise RuntimeError(m)
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(p,v):p.write_text(json.dumps(v,indent=2,sort_keys=True,default=str))
def read_json(p):return json.loads(Path(p).read_text())
def image_map(folder):
    out={};folder=Path(folder)
    if folder.exists():
        for p in folder.iterdir():
            if p.is_file() and p.suffix.lower() in SUPPORTED:
                require(p.stem not in out,f'Duplicate image stem {p.stem}');out[p.stem]=p
    return out
def newest_direct_complete(roots,required_groups):
    hits=[]
    for root in roots:
        if not Path(root).exists():continue
        for p in Path(root).iterdir():
            if p.is_dir() and all(any((p/r).exists() for r in group) for group in required_groups):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def resolve_recorded_path(value,bases):
    if value is None or str(value).strip() in {'','None','nan'}:return None
    p=Path(str(value));candidates=[p]+[Path(b)/p for b in bases]+[Path(b)/p.name for b in bases]
    hits=[]
    for q in candidates:
        if q.exists() and q.resolve() not in hits:hits.append(q.resolve())
    require(len(hits)<=1,f'Ambiguous path {value}: {hits}')
    return hits[0] if hits else None
def unique_paths(values,bases,description):
    hits=[]
    for v in values:
        q=resolve_recorded_path(v,bases)
        if q is not None and q not in hits:hits.append(q)
    require(len(hits)==1,f'Expected one {description}; found {hits}')
    return hits[0]
def bounded_named(root,names,max_depth=3):
    hits=[];root=Path(root)
    for current,dirs,files in os.walk(root):
        if len(Path(current).relative_to(root).parts)>=max_depth:dirs[:]=[]
        for name in files:
            if name in names:hits.append(Path(current)/name)
    return hits

## 3. Pin Notebook 70, Pilot 09 and Notebook 53; enforce exact roster equality

In [ ]:
N70=Path(NOTEBOOK70_RUN_OVERRIDE) if NOTEBOOK70_RUN_OVERRIDE else newest_direct_complete(N70_ROOTS,[['tables/metrics_by_page_arm.csv','tables/whole_page_metrics.csv','tables/page_metrics.csv'],['notebook70_verdict.json','verdict.json','executive_summary.txt']])
require(N70 is not None and N70.is_dir(),'Set NOTEBOOK70_RUN_OVERRIDE to the completed Notebook 70 run.')
roster_candidates=[N70/'inputs/page_roster.csv',N70/'tables/page_roster.csv',N70/'tables/resolved_page_roster.csv',N70/'tables/metrics_by_page_arm.csv',N70/'tables/whole_page_metrics.csv',N70/'tables/page_metrics.csv']
roster_hits=[p for p in roster_candidates if p.exists()];require(roster_hits,f'No roster table in {N70}')
roster_source=roster_hits[0];rt=pd.read_csv(roster_source,dtype={'page':str});require('page' in rt.columns,f'{roster_source} lacks page')
pages=sorted(rt.page.astype(str).dropna().unique());require(pages,'Empty Notebook 70 roster')
source={}
for p in [N70/'source_record.json',N70/'experiment_identity.json',N70/'audit/source_record.json']:
    if p.exists():source.update(read_json(p))
def recorded(keys):
    for k in keys:
        if source.get(k):return Path(str(source[k]))
    return None
r09=recorded(['notebook09_run','pilot09_run','notebook_09_run','n09']);r53=recorded(['notebook53_run','v88_run','notebook53R3_run','n53'])
N09=Path(NOTEBOOK09_RUN_OVERRIDE) if NOTEBOOK09_RUN_OVERRIDE else (r09 if r09 is not None and r09.exists() else newest_direct_complete([N09_ROOT],[['pilot_manifest.json'],['tables/component_evidence.csv'],['proposals']]))
N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else (r53 if r53 is not None and r53.exists() else newest_direct_complete(N53_ROOTS,[['experiment_identity.json'],['pages']]))
require(N09 is not None and N09.is_dir(),'Set NOTEBOOK09_RUN_OVERRIDE to the exact Pilot 09 run.')
require(N53 is not None and N53.is_dir(),'Set NOTEBOOK53_RUN_OVERRIDE to the exact Notebook 53 run.')
component_path=N09/'tables/component_evidence.csv';require(component_path.exists(),f'Missing {component_path}')
components=pd.read_csv(component_path,dtype={'page':str},low_memory=False);roster09=sorted(components.page.astype(str).unique())
roster53=sorted(p.name for p in (N53/'pages').iterdir() if p.is_dir())
audit={'notebook70_run':str(N70),'roster_source':str(roster_source),'notebook09_run':str(N09),'notebook53_run':str(N53),'pages70':len(pages),'pages09':len(roster09),'pages53':len(roster53),'only70_vs09':sorted(set(pages)-set(roster09)),'only09_vs70':sorted(set(roster09)-set(pages)),'only70_vs53':sorted(set(pages)-set(roster53)),'only53_vs70':sorted(set(roster53)-set(pages)),'exact70_09':list(pages)==list(roster09),'exact70_53':list(pages)==list(roster53)}
atomic_json(AUDIT/'roster_audit.json',audit);require(audit['exact70_09'],f'Notebook70/Pilot09 roster mismatch: {audit}');require(audit['exact70_53'],f'Notebook70/Notebook53 roster mismatch: {audit}')
pd.DataFrame({'page':pages}).to_csv(INPUTS/'exact_page_roster.csv',index=False)
print('N70:',N70);print('N09:',N09);print('N53:',N53);print('Exact pages:',len(pages))

## 4. Resolve exact Pilot 09 observed pages and reviewed seed masks

In [ ]:
resolution_files=sorted((N09/'audit').glob('*_input_resolution.json'));require(resolution_files,'No Pilot 09 input-resolution records')
records={}
for p in resolution_files:
    r=read_json(p);page=str(r.get('page') or p.name.replace('_input_resolution.json',''));require(page not in records,f'Duplicate input record {page}');records[page]=(p,r)
require(set(records)==set(pages),f'Input-resolution roster mismatch: missing={sorted(set(pages)-set(records))}, extra={sorted(set(records)-set(pages))}')
obs_keys=['observed_path','observed_image_path','aligned_output','page_path','source_path','source_file'];mask_keys=['reviewed_mask_path','corrected_mask_output','mask_path','authoritative_mask_path','seed_mask_path']
RESOLVED={};rows=[]
for page in pages:
    recpath,r=records[page];bases=[N09,N09/'inputs',N09/'masks',N09/'audit',ROOT,ROOT/'aligned',ROOT/'source/annotated_only',ROOT/'production_pilot/reviewed_masks',ROOT/'masks/corrected']
    obs=unique_paths([r.get(k) for k in obs_keys if r.get(k)],bases,f'{page} observed');seed=unique_paths([r.get(k) for k in mask_keys if r.get(k)],bases,f'{page} reviewed mask')
    o=cv2.imread(str(obs),0);m=cv2.imread(str(seed),0);require(o is not None and m is not None and o.shape==m.shape,f'{page}: unreadable or shape mismatch')
    RESOLVED[page]={'observed':obs,'seed':seed,'record':recpath};rows.append({'page':page,'record':str(recpath),'observed_path':str(obs),'observed_sha256':sha256_file(obs),'seed_path':str(seed),'seed_sha256':sha256_file(seed),'height':o.shape[0],'width':o.shape[1]})
pd.DataFrame(rows).to_csv(AUDIT/'resolved_pilot09_inputs.csv',index=False);print('Resolved:',len(rows))

## 5. Canonical Notebook 09 functions

In [ ]:
def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)
def prepare_mask(seed,dilation=MASK_DILATION):
    mask=binary(seed)
    if dilation>0:mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
    return mask
def immutable(original,candidate,mask):
    out=original.copy();out[mask>0]=candidate[mask>0];require(np.array_equal(out[mask==0],original[mask==0]),'outside-mask drift');return out
def padded_components(mask):
    n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
    for i in range(1,n):
        x,y,bw,bh,area=stats[i]
        if area>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
    changed=True
    while changed:
        changed=False;merged=[]
        while boxes:
            ax0,ay0,ax1,ay1=boxes.pop();rest=[]
            for bx0,by0,bx1,by1 in boxes:
                if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
                else:rest.append((bx0,by0,bx1,by1))
            boxes=rest;merged.append((ax0,ay0,ax1,ay1))
        boxes=merged
    return boxes
def telea_reconstruct(observed,mask):
    result=observed.copy()
    for x0,y0,x1,y1 in padded_components(mask):
        im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
    return immutable(observed,result,mask)
def background_surface(observed,mask,sigma=45.0):
    visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32);num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma);return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def support_maps(observed,mask):
    dark=(observed<190)&(mask==0);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0;n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,hg,a=stats[i]
        if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
    k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8);return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0
def geometry_t80(observed,mask,telea):
    paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0;preserve=m&(horizontal|vertical|compact)&(observed<200);remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD);ambiguous=m&~preserve&~remove;result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve];return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous}
CANONICAL_STRUCTURAL_BLOCK_SHA256='cce24d65e8866b086956855b55e9a5104383c8b936629c5035354968c929b04c'
print('Notebook09 canonical block:',CANONICAL_STRUCTURAL_BLOCK_SHA256)

## 6. Resolve frozen V8.8 artefacts

In [ ]:
def page_artifact(page_dir,names,label):
    hits=[(page_dir/n).resolve() for n in names if (page_dir/n).exists()]
    if not hits:hits=[p.resolve() for p in bounded_named(page_dir,names)]
    hits=list(dict.fromkeys(hits));require(len(hits)==1,f'{page_dir.name}: expected one {label}; found {hits}');return hits[0]
V88={};rows=[]
for page in pages:
    d=N53/'pages'/page;require(d.exists(),f'{page}: no Notebook53 page folder')
    m3p=page_artifact(d,['m3.png','M3.png','authoritative_mask.png'],'M3');sp=page_artifact(d,['safe_remove.png','SAFE_REMOVE.png','safe_remove_mask.png'],'SAFE_REMOVE');m5p=page_artifact(d,['m5_remove.png','M5_REMOVE.png','m5.png','residual_mask.png'],'M5_REMOVE');cp=page_artifact(d,['corrected.png','v8_8_corrected.png','registered_replay_corrected.png'],'corrected page')
    m3=cv2.imread(str(m3p),0)>0;s=cv2.imread(str(sp),0)>0;m5=cv2.imread(str(m5p),0)>0;require(m3.shape==s.shape==m5.shape,f'{page}: V8.8 mask shape mismatch');require(np.all(~s|m3),f'{page}: SAFE_REMOVE outside M3');require(np.array_equal(m5,m3&~s),f'{page}: M5 != M3 minus SAFE_REMOVE')
    V88[page]={'m3':m3p,'safe':sp,'m5':m5p,'corrected':cp};rows.append({'page':page,'m3_path':str(m3p),'safe_remove_path':str(sp),'m5_path':str(m5p),'corrected_path':str(cp),'m3_pixels':int(m3.sum()),'safe_remove_pixels':int(s.sum()),'m5_pixels':int(m5.sum()),'m3_hash':array_hash(m3.astype(np.uint8)),'safe_hash':array_hash(s.astype(np.uint8)),'m5_hash':array_hash(m5.astype(np.uint8))})
pd.DataFrame(rows).to_csv(AUDIT/'resolved_v88_artifacts.csv',index=False);print('V8.8 pages:',len(rows))

## 7. Fresh engine rerun and pre-truth output lock

In [ ]:
rows=[];failures=[]
for number,page in enumerate(pages,1):
    try:
        obs=cv2.imread(str(RESOLVED[page]['observed']),0);seed=cv2.imread(str(RESOLVED[page]['seed']),0);m3=prepare_mask(seed)>0;fm3=cv2.imread(str(V88[page]['m3']),0)>0;safe=cv2.imread(str(V88[page]['safe']),0)>0;m5=cv2.imread(str(V88[page]['m5']),0)>0
        require(obs is not None and seed is not None and obs.shape==seed.shape==m3.shape,f'{page}: input shape failure');require(np.array_equal(m3,fm3),f'{page}: Pilot09 M3 differs from V8.8 M3');require(np.array_equal(m5,m3&~safe),f'{page}: M5 identity failure')
        telea=telea_reconstruct(obs,m3.astype(np.uint8)*255);n09,_=geometry_t80(obs,m3.astype(np.uint8)*255,telea);v88=telea_reconstruct(obs,m5.astype(np.uint8)*255)
        reg88=cv2.imread(str(V88[page]['corrected']),0);require(reg88 is not None and np.array_equal(v88,reg88),f'{page}: fresh V8.8 replay mismatch')
        props=[p for p in [N09/'proposals'/f'{page}.png',N09/'proposals'/f'{page}_proposal.png'] if p.exists()];require(len(props)==1,f'{page}: Pilot09 proposal not unique: {props}');reg09=cv2.imread(str(props[0]),0);require(reg09 is not None and np.array_equal(n09,reg09),f'{page}: fresh Notebook09 replay mismatch')
        require(np.array_equal(n09[~m3],obs[~m3]),f'{page}: N09 outside drift');require(np.array_equal(v88[~m3],obs[~m3]),f'{page}: V8.8 outside drift');require(np.array_equal(v88[safe],obs[safe]),f'{page}: SAFE_REMOVE not observed')
        d=PAGES_OUT/page;d.mkdir();arrays={'observed':obs,'reviewed_seed_mask':binary(seed),'authoritative_m3':m3.astype(np.uint8)*255,'safe_remove':safe.astype(np.uint8)*255,'m5_remove':m5.astype(np.uint8)*255,'notebook09_fresh':n09,'v88_fresh':v88}
        for name,a in arrays.items():require(cv2.imwrite(str(d/f'{name}.png'),a),f'{page}: write {name} failed')
        rows.append({'page':page,'height':obs.shape[0],'width':obs.shape[1],'m3_pixels':int(m3.sum()),'safe_remove_pixels':int(safe.sum()),'m5_pixels':int(m5.sum()),'notebook09_changed_pixels':int((n09!=obs).sum()),'v88_changed_pixels':int((v88!=obs).sum()),'engine_difference_pixels':int((n09!=v88).sum()),'notebook09_outside_m3_drift':int((n09[~m3]!=obs[~m3]).sum()),'v88_outside_m3_drift':int((v88[~m3]!=obs[~m3]).sum()),'observed_sha256':array_hash(obs),'m3_sha256':array_hash(m3.astype(np.uint8)),'safe_remove_sha256':array_hash(safe.astype(np.uint8)),'m5_sha256':array_hash(m5.astype(np.uint8)),'notebook09_sha256':array_hash(n09),'v88_sha256':array_hash(v88),'exact_notebook09_replay':True,'exact_v88_replay':True})
        print(f'{number}/{len(pages)} {page}: difference={int((n09!=v88).sum()):,}')
    except Exception as e:failures.append({'page':page,'error':repr(e)});print('FAILED',page,repr(e))
cols=['page','height','width','m3_pixels','safe_remove_pixels','m5_pixels','notebook09_changed_pixels','v88_changed_pixels','engine_difference_pixels','notebook09_outside_m3_drift','v88_outside_m3_drift','observed_sha256','m3_sha256','safe_remove_sha256','m5_sha256','notebook09_sha256','v88_sha256','exact_notebook09_replay','exact_v88_replay']
engines=pd.DataFrame(rows,columns=cols);fail=pd.DataFrame(failures,columns=['page','error']);engines.to_csv(TABLES/'fresh_engine_inventory.csv',index=False);fail.to_csv(TABLES/'failures.csv',index=False)
require(fail.empty,f'Engine failures: see {TABLES/"failures.csv"}');require(engines.page.astype(str).tolist()==list(pages),'Completed roster/order mismatch');require(int(engines.notebook09_outside_m3_drift.sum())==0 and int(engines.v88_outside_m3_drift.sum())==0,'Outside-M3 drift')
lock={'created_utc':datetime.now(timezone.utc).isoformat(),'notebook70':str(N70),'notebook09':str(N09),'notebook53':str(N53),'ordered_pages':list(pages),'roster_sha256':object_hash(list(pages)),'engine_inventory_sha256':sha256_file(TABLES/'fresh_engine_inventory.csv'),'outputs':engines[['page','observed_sha256','m3_sha256','safe_remove_sha256','m5_sha256','notebook09_sha256','v88_sha256']].to_dict('records'),'clean_truth_loaded':False};atomic_json(AUDIT/'candidate_output_lock_before_truth.json',lock);print('Locked outputs:',len(engines))

## 8. Clean-truth unlock

In [ ]:
clean_map=image_map(CLEAN_DIR);missing=sorted(set(pages)-set(clean_map));atomic_json(AUDIT/'clean_truth_roster_audit.json',{'directory':str(CLEAN_DIR),'missing':missing,'extra_ignored':sorted(set(clean_map)-set(pages))});require(not missing,f'Missing clean pages: {missing}')
rows=[]
for page in pages:
    c=cv2.imread(str(clean_map[page]),0);o=cv2.imread(str(PAGES_OUT/page/'observed.png'),0);require(c is not None and o is not None and c.shape==o.shape,f'{page}: clean shape mismatch');rows.append({'page':page,'clean_path':str(clean_map[page]),'clean_sha256':sha256_file(clean_map[page]),'height':c.shape[0],'width':c.shape[1]})
pd.DataFrame(rows).to_csv(AUDIT/'resolved_clean_truth.csv',index=False);print('Clean pages:',len(rows))

## 9. Whole-page and inside-M3 metrics

In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def prf(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region;k=np.ones((2*tol+1,2*tol+1),np.uint8);rd=cv2.dilate(ref.astype(np.uint8),k)>0;pdil=cv2.dilate(pred.astype(np.uint8),k)>0;p=float((pred&rd).sum()/max(int(pred.sum()),1));r=float((ref&pdil).sum()/max(int(ref.sum()),1));return p,r,2*p*r/max(p+r,1e-12)
def score(clean,cand,region):
    ci=printed_ink(clean);pi=printed_ink(cand);d=clean.astype(np.float32)-cand.astype(np.float32);_,sm=structural_similarity(clean,cand,data_range=255,full=True);ip,ir,if1=prf(ci,pi,region);ep,er,ef1=prf(edges(clean),edges(cand),region);fi=int((pi&~ci&region).sum());mi=int((ci&~pi&region).sum());mse=float((d[region]**2).mean());return {'pixels':int(region.sum()),'mae':float(np.abs(d[region]).mean()),'rmse':float(math.sqrt(mse)),'ssim':float(sm[region].mean()),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'false_inserted_ink':fi,'false_inserted_rate':float(fi/max(int(region.sum()),1)),'missing_clean_ink':mi,'missing_clean_rate':float(mi/max(int(ci[region].sum()),1))}
rows=[]
for page in pages:
    c=cv2.imread(str(clean_map[page]),0);n=cv2.imread(str(PAGES_OUT/page/'notebook09_fresh.png'),0);v=cv2.imread(str(PAGES_OUT/page/'v88_fresh.png'),0);m=cv2.imread(str(PAGES_OUT/page/'authoritative_m3.png'),0)>0;whole=np.ones(c.shape,bool)
    for engine,cand in [('NOTEBOOK09',n),('V8_8',v)]:
        for scope,region in [('WHOLE_PAGE',whole),('INSIDE_M3',m)]:rows.append({'page':page,'engine':engine,'scope':scope,**score(c,cand,region)})
metrics=pd.DataFrame(rows);metrics.to_csv(TABLES/'clean_truth_metrics_by_page_engine_scope.csv',index=False);require(len(metrics)==4*len(pages),'Incomplete metrics');display(metrics.groupby(['scope','engine'])[['mae','ssim','ink_f1','edge_f1','false_inserted_ink','missing_clean_ink']].mean())

## 10. Page-paired effects and bootstrap confidence intervals

In [ ]:
directions={'mae':'lower','rmse':'lower','ssim':'higher','ink_precision':'higher','ink_recall':'higher','ink_f1':'higher','edge_precision':'higher','edge_recall':'higher','edge_f1':'higher','false_inserted_ink':'lower','false_inserted_rate':'lower','missing_clean_ink':'lower','missing_clean_rate':'lower'}
def bootstrap(v,seed):
    v=np.asarray(v,float);v=v[np.isfinite(v)];rng=np.random.default_rng(seed);draw=np.array([rng.choice(v,len(v),replace=True).mean() for _ in range(BOOTSTRAP_RESAMPLES)]);return float(v.mean()),float(np.percentile(draw,2.5)),float(np.percentile(draw,97.5))
paired=[];summ=[]
for scope in ['WHOLE_PAGE','INSIDE_M3']:
    sub=metrics[metrics.scope==scope]
    for metric,direction in directions.items():
        w=sub.pivot(index='page',columns='engine',values=metric).loc[pages];raw=w.V8_8-w.NOTEBOOK09;benefit=raw if direction=='higher' else -raw;mean,lo,hi=bootstrap(benefit,SEED+len(summ))
        for page in pages:paired.append({'page':page,'scope':scope,'metric':metric,'direction':direction,'notebook09':float(w.loc[page,'NOTEBOOK09']),'v8_8':float(w.loc[page,'V8_8']),'v8_8_minus_notebook09':float(raw.loc[page]),'benefit_positive_is_v8_8_better':float(benefit.loc[page])})
        summ.append({'scope':scope,'metric':metric,'direction':direction,'pages':len(benefit),'mean_benefit':mean,'ci95_low':lo,'ci95_high':hi,'median_benefit':float(np.median(benefit)),'wins':int((benefit>0).sum()),'losses':int((benefit<0).sum()),'ties':int((benefit==0).sum())})
paired=pd.DataFrame(paired);summary=pd.DataFrame(summ);paired.to_csv(TABLES/'paired_page_effects.csv',index=False);summary.to_csv(TABLES/'paired_page_bootstrap_summary.csv',index=False);display(summary[(summary.scope=='WHOLE_PAGE')&summary.metric.isin(['missing_clean_ink','false_inserted_ink','ink_f1','edge_f1','mae','ssim'])])

## 11. Whole-page visual audit

In [ ]:
def heat(a):
    a=np.asarray(a,np.float32);return np.zeros(a.shape,np.uint8) if a.max()<=0 else np.clip(255*a/a.max(),0,255).astype(np.uint8)
def render(page,label):
    o=cv2.imread(str(PAGES_OUT/page/'observed.png'),0);c=cv2.imread(str(clean_map[page]),0);n=cv2.imread(str(PAGES_OUT/page/'notebook09_fresh.png'),0);v=cv2.imread(str(PAGES_OUT/page/'v88_fresh.png'),0);m=cv2.imread(str(PAGES_OUT/page/'authoritative_m3.png'),0);fig,ax=plt.subplots(2,4,figsize=(20,12));panels=[(o,'Observed'),(c,'Clean truth'),(n,'Fresh Notebook 09'),(v,'Fresh V8.8'),(m,'Authoritative M3'),(heat(np.abs(n.astype(np.int16)-c.astype(np.int16))),'N09 error'),(heat(np.abs(v.astype(np.int16)-c.astype(np.int16))),'V8.8 error'),(heat(np.abs(v.astype(np.int16)-n.astype(np.int16))),'Engine difference')]
    for a,(im,t) in zip(ax.ravel(),panels):a.imshow(im,cmap='gray',vmin=0,vmax=255);a.set_title(t);a.axis('off')
    fig.suptitle(f'{label} | {page}',fontsize=16);plt.tight_layout();p=VISUAL/f'{label}_{page}.png';fig.savefig(p,dpi=150,bbox_inches='tight');plt.close(fig);return p
w=paired[paired.scope=='WHOLE_PAGE'];mi=w[w.metric=='missing_clean_ink'].set_index('page').benefit_positive_is_v8_8_better;fi=w[w.metric=='false_inserted_ink'].set_index('page').benefit_positive_is_v8_8_better;rank=pd.DataFrame({'page':pages,'missing_ink_benefit':[mi[p] for p in pages],'false_inserted_benefit':[fi[p] for p in pages]});rank['balanced_primary_benefit']=rank.missing_ink_benefit+rank.false_inserted_benefit;rank=rank.sort_values(['balanced_primary_benefit','page'],ascending=[False,True]);rank.to_csv(TABLES/'visual_review_ranking.csv',index=False)
selected=[]
for label,frame in [('BEST',rank.head(MAX_VISUAL_PAGES//2)),('WORST',rank.tail(MAX_VISUAL_PAGES//2).sort_values('balanced_primary_benefit'))]:
    for i,r in frame.reset_index(drop=True).iterrows():selected.append({'page':r.page,'category':label,'rank':i+1,'path':str(render(r.page,f'{label}_{i+1:02d}')),'missing_ink_benefit':r.missing_ink_benefit,'false_inserted_benefit':r.false_inserted_benefit,'balanced_primary_benefit':r.balanced_primary_benefit})
pd.DataFrame(selected).to_csv(TABLES/'visual_review_manifest.csv',index=False);print('Rendered:',len(selected))

## 12. Verdict and upload bundle

In [ ]:
def q(scope,metric):
    x=summary[(summary.scope==scope)&(summary.metric==metric)]
    require(len(x)==1,f'Missing {scope}/{metric}')
    return x.iloc[0]
wm=q('WHOLE_PAGE','missing_clean_ink');wf=q('WHOLE_PAGE','false_inserted_ink');mm=q('INSIDE_M3','missing_clean_ink');mf=q('INSIDE_M3','false_inserted_ink')
checks={
'exact_notebook70_notebook09_roster':audit['exact70_09'],'exact_notebook70_notebook53_roster':audit['exact70_53'],
'all_pages_completed':len(engines)==len(pages),'no_engine_failures':fail.empty,
'exact_notebook09_replay':bool(engines.exact_notebook09_replay.all()),'exact_v88_replay':bool(engines.exact_v88_replay.all()),
'zero_notebook09_outside_m3_drift':int(engines.notebook09_outside_m3_drift.sum())==0,
'zero_v88_outside_m3_drift':int(engines.v88_outside_m3_drift.sum())==0,'complete_clean_truth':not missing,
'whole_page_missing_ink_reduced_ci':float(wm.ci95_low)>0,'whole_page_false_inserted_not_increased_ci':float(wf.ci95_low)>=0,
'inside_m3_missing_ink_reduced_ci':float(mm.ci95_low)>0,'inside_m3_false_inserted_not_increased_ci':float(mf.ci95_low)>=0}
integrity_names=['exact_notebook70_notebook09_roster','exact_notebook70_notebook53_roster','all_pages_completed','no_engine_failures','exact_notebook09_replay','exact_v88_replay','zero_notebook09_outside_m3_drift','zero_v88_outside_m3_drift','complete_clean_truth']
integrity=all(checks[k] for k in integrity_names)
if not integrity:
    verdict='INVALID_RUN_INTEGRITY_OR_REPLAY_FAILURE'
elif checks['whole_page_missing_ink_reduced_ci'] and checks['whole_page_false_inserted_not_increased_ci'] and checks['inside_m3_missing_ink_reduced_ci'] and checks['inside_m3_false_inserted_not_increased_ci']:
    verdict='PASS_V88_REDUCES_MISSING_PRINT_WITHOUT_RESIDUAL_ANNOTATION_REGRESSION'
elif wm.mean_benefit>0 and wf.mean_benefit>=0:
    verdict='AMBER_POINT_ESTIMATE_FAVOURS_V88_BUT_UNCERTAINTY_INCONCLUSIVE'
elif wm.mean_benefit>0 and wf.mean_benefit<0:
    verdict='AMBER_V88_PRINT_PRESERVATION_GAIN_WITH_RESIDUAL_ANNOTATION_TRADEOFF'
else:
    verdict='NO_EVIDENCE_V88_BEATS_NOTEBOOK09_ON_PRIMARY_CLEAN_TRUTH_PAIR'
record={'notebook':'71R1','created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'pages':len(pages),'checks':checks,
'primary_whole_page':{'missing_clean_ink_mean_benefit':float(wm.mean_benefit),'missing_clean_ink_ci95':[float(wm.ci95_low),float(wm.ci95_high)],'false_inserted_ink_mean_benefit':float(wf.mean_benefit),'false_inserted_ink_ci95':[float(wf.ci95_low),float(wf.ci95_high)]},
'secondary_inside_m3':{'missing_clean_ink_mean_benefit':float(mm.mean_benefit),'missing_clean_ink_ci95':[float(mm.ci95_low),float(mm.ci95_high)],'false_inserted_ink_mean_benefit':float(mf.mean_benefit),'false_inserted_ink_ci95':[float(mf.ci95_low),float(mf.ci95_high)]},
'interpretation':'Positive benefit means V8.8 is better. One page contributes once. Both paths share M3; V8.8 only subtracts SAFE_REMOVE.','production_authority':False}
atomic_json(OUT/'notebook71R1_verdict.json',record)
lines=[]
lines.append('Notebook 71R1 - exact-roster whole-page engine comparison')
lines.append('')
lines.append(f'Verdict: {verdict}')
lines.append(f'Pages: {len(pages)}')
lines.append(f'Notebook 70: {N70}')
lines.append(f'Pilot 09: {N09}')
lines.append(f'Notebook 53: {N53}')
lines.append('')
lines.append('Positive benefit means V8.8 is better.')
lines.append(f'Whole-page missing-clean-ink benefit: {wm.mean_benefit:.6f}, CI [{wm.ci95_low:.6f}, {wm.ci95_high:.6f}]')
lines.append(f'Whole-page false-inserted-ink benefit: {wf.mean_benefit:.6f}, CI [{wf.ci95_low:.6f}, {wf.ci95_high:.6f}]')
lines.append(f'Inside-M3 missing-clean-ink benefit: {mm.mean_benefit:.6f}, CI [{mm.ci95_low:.6f}, {mm.ci95_high:.6f}]')
lines.append(f'Inside-M3 false-inserted-ink benefit: {mf.mean_benefit:.6f}, CI [{mf.ci95_low:.6f}, {mf.ci95_high:.6f}]')
lines.append('')
lines.append('Both engines were freshly rebuilt and exactly replayed before clean truth was loaded. No production authority.')
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
atomic_json(AUDIT/'software_environment.json',{'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'opencv':cv2.__version__})
files=[OUT/'notebook71R1_verdict.json',OUT/'executive_summary.txt',TABLES/'fresh_engine_inventory.csv',TABLES/'failures.csv',TABLES/'clean_truth_metrics_by_page_engine_scope.csv',TABLES/'paired_page_effects.csv',TABLES/'paired_page_bootstrap_summary.csv',TABLES/'visual_review_ranking.csv',TABLES/'visual_review_manifest.csv',AUDIT/'roster_audit.json',AUDIT/'resolved_pilot09_inputs.csv',AUDIT/'resolved_v88_artifacts.csv',AUDIT/'candidate_output_lock_before_truth.json',AUDIT/'clean_truth_roster_audit.json',AUDIT/'resolved_clean_truth.csv',AUDIT/'software_environment.json',INPUTS/'exact_page_roster.csv']
for item in files:
    require(item.exists(),f'Missing upload file {item}')
    shutil.copy2(item,UPLOAD/item.name)
for item in VISUAL.glob('*.png'):
    shutil.copy2(item,UPLOAD/item.name)
inv=[]
for item in sorted(UPLOAD.iterdir()):
    if item.is_file():inv.append({'file':item.name,'bytes':item.stat().st_size,'sha256':sha256_file(item)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=OUT/'Notebook71R1_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(UPLOAD.iterdir()):
        if item.is_file():archive.write(item,item.name)
print(chr(10).join(lines))
print('Visual audit:',VISUAL)
print('ZIP:',zip_path)
